In [2]:
import dlib
import numpy as np
import pandas as pd
import sqlalchemy as db
from qdrant_client import QdrantClient 
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client.http import models

In [3]:
CLIENT = QdrantClient(host='192.168.0.131', port=6333)   

In [4]:
username = 'amos'
password = 'M0$hicat'
host = '192.168.0.131'
port = '3306'
database = 'CineFace'
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
engine = db.create_engine(connection_string)
conn = engine.connect()

In [5]:
df = pd.read_sql_query('SELECT * FROM faces WHERE series_id = 1442437;', conn)
df.head()

,episode_id,frame_num,face_num,series_id,season,episode,filename,x1,y1,x2,...,mouth_left_y,confidence,img_height,img_width,pct_of_frame,distance_from_center,encoding_id,cast_id,encoding,filepath
0,64952,48,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.469,0.115,0.597,...,0.361,0.999,1080,1920,0.043264,242.33,2ecc0db5-8326-423d-9127-b476a9c5b011,None,-0.0206817\n0.105229\n0.0132743\n-0.0701414\n-...,/home/amos/media/tv/modern_family/Modern.Famil...
1,64952,72,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.471,0.145,0.602,...,0.406,1.000,1080,1920,0.045719,207.18,ccd4c9c0-5dd8-497a-a4ab-836e94d74039,None,-0.0940071\n0.124249\n0.0446877\n-0.0341553\n-...,/home/amos/media/tv/modern_family/Modern.Famil...
2,64952,96,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.470,0.113,0.598,...,0.348,1.000,1080,1920,0.042368,248.65,0ef86dc4-eb1f-44df-982a-be4af2f37050,None,-0.0610968\n0.102561\n0.0062373\n-0.0656137\n-...,/home/amos/media/tv/modern_family/Modern.Famil...
3,64952,120,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.450,0.118,0.579,...,0.355,1.000,1080,1920,0.043215,233.57,8522a7f2-f369-48c6-9e4f-1cfd2ffc60eb,None,-0.0373278\n0.136002\n-0.00652923\n-0.0617118\...,/home/amos/media/tv/modern_family/Modern.Famil...
4,64952,144,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.432,0.135,0.565,...,0.369,1.000,1080,1920,0.044422,214.02,1360f492-1447-45fc-b970-378b91874f15,None,-0.0639436\n0.129092\n0.0101053\n-0.0582337\n-...,/home/amos/media/tv/modern_family/Modern.Famil...


In [6]:
episode_df = df[df['episode_id'] == 64952]
episode_df.head()

,episode_id,frame_num,face_num,series_id,season,episode,filename,x1,y1,x2,...,mouth_left_y,confidence,img_height,img_width,pct_of_frame,distance_from_center,encoding_id,cast_id,encoding,filepath
0,64952,48,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.469,0.115,0.597,...,0.361,0.999,1080,1920,0.043264,242.33,2ecc0db5-8326-423d-9127-b476a9c5b011,None,-0.0206817\n0.105229\n0.0132743\n-0.0701414\n-...,/home/amos/media/tv/modern_family/Modern.Famil...
1,64952,72,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.471,0.145,0.602,...,0.406,1.000,1080,1920,0.045719,207.18,ccd4c9c0-5dd8-497a-a4ab-836e94d74039,None,-0.0940071\n0.124249\n0.0446877\n-0.0341553\n-...,/home/amos/media/tv/modern_family/Modern.Famil...
2,64952,96,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.470,0.113,0.598,...,0.348,1.000,1080,1920,0.042368,248.65,0ef86dc4-eb1f-44df-982a-be4af2f37050,None,-0.0610968\n0.102561\n0.0062373\n-0.0656137\n-...,/home/amos/media/tv/modern_family/Modern.Famil...
3,64952,120,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.450,0.118,0.579,...,0.355,1.000,1080,1920,0.043215,233.57,8522a7f2-f369-48c6-9e4f-1cfd2ffc60eb,None,-0.0373278\n0.136002\n-0.00652923\n-0.0617118\...,/home/amos/media/tv/modern_family/Modern.Famil...
4,64952,144,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.432,0.135,0.565,...,0.369,1.000,1080,1920,0.044422,214.02,1360f492-1447-45fc-b970-378b91874f15,None,-0.0639436\n0.129092\n0.0101053\n-0.0582337\n-...,/home/amos/media/tv/modern_family/Modern.Famil...


In [11]:
encodings = episode_df['encoding'].map(lambda x: dlib.vector(np.array([float(y) for y in x.split('\n')])))
encodings[0]

dlib.vector([-0.0206817, 0.105229, 0.0132743, -0.0701414, -0.12959, -0.077303, -0.0151558, -0.135242, 0.15644, -0.0654795, 0.181302, -0.0084787, -0.203364, -0.0108257, -0.0893914, 0.0251591, -0.146696, -0.0803495, -0.122477, -0.141012, 0.0142192, 0.0437388, 0.07803, 0.021838, -0.128347, -0.211577, -0.0587878, -0.10156, 0.142533, -0.0857485, -0.0241423, 0.0805529, -0.283509, -0.152032, 0.0130118, -0.00784918, -0.0567647, -0.0306636, 0.257818, 0.0665399, -0.11678, 0.0500921, 0.0364346, 0.277925, 0.222498, 0.0428025, 0.00728985, -0.0557249, 0.166578, -0.260112, 0.032655, 0.154734, 0.169613, 0.0300287, 0.105058, -0.219169, 0.0412286, 0.0821967, -0.201235, 0.150301, 0.0556454, -0.0331962, 0.00572758, -0.0370779, 0.178074, 0.06963, -0.0463299, -0.101108, 0.166423, -0.153209, 0.0138914, 0.174855, -0.0803663, -0.144038, -0.222853, -0.00715435, 0.44084, 0.185731, -0.0917033, -0.00533417, -0.104421, -0.0685382, 0.0394521, -0.0391319, -0.16553, 0.0103078, -0.0556923, 0.0997629, 0.202144, 0.003319

In [19]:
labels = dlib.chinese_whispers_clustering(list(encodings), 0.35)

In [33]:
label_df = episode_df.copy()
label_df['label'] = labels
cnt = label_df['label'].value_counts()
label_df = label_df.merge(cnt,
                          on='label')
label_df = label_df[label_df['count'] >= 5]
label_df.head()

,episode_id,frame_num,face_num,series_id,season,episode,filename,x1,y1,x2,...,img_height,img_width,pct_of_frame,distance_from_center,encoding_id,cast_id,encoding,filepath,label,count
1,64952,72,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.471,0.145,0.602,...,1080,1920,0.045719,207.18,ccd4c9c0-5dd8-497a-a4ab-836e94d74039,None,-0.0940071\n0.124249\n0.0446877\n-0.0341553\n-...,/home/amos/media/tv/modern_family/Modern.Famil...,1,7
2,64952,96,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.470,0.113,0.598,...,1080,1920,0.042368,248.65,0ef86dc4-eb1f-44df-982a-be4af2f37050,None,-0.0610968\n0.102561\n0.0062373\n-0.0656137\n-...,/home/amos/media/tv/modern_family/Modern.Famil...,1,7
4,64952,144,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.432,0.135,0.565,...,1080,1920,0.044422,214.02,1360f492-1447-45fc-b970-378b91874f15,None,-0.0639436\n0.129092\n0.0101053\n-0.0582337\n-...,/home/amos/media/tv/modern_family/Modern.Famil...,1,7
5,64952,168,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.446,0.131,0.580,...,1080,1920,0.045292,217.33,53c6b889-093a-4841-8785-06a2f125ef9e,None,-0.0634293\n0.14206\n0.00504685\n-0.0659082\n-...,/home/amos/media/tv/modern_family/Modern.Famil...,1,7
6,64952,192,0,1442437,1,18,Modern.Family.S01E18.1080p.BluRay.x265-RARBG.mp4,0.453,0.128,0.584,...,1080,1920,0.043754,224.74,c79ce411-6dd7-4e95-9b24-93311344f127,None,-0.0835662\n0.131924\n0.0267161\n-0.0545587\n-...,/home/amos/media/tv/modern_family/Modern.Famil...,1,7


In [34]:
label_df.to_csv('../../data/clusters/modern_family/S01E18.csv')

In [35]:
episode_df.shape[0]

2124